# KV260 DPU Full Benchmark
**Platform:** AMD Kria KV260 · DPU B512 (DPUCZDX8G) · XRT 2.13.479 · pynq-dpu 2.5  
**Methodology:** 10 rounds × 100 frames · 60s cool-down between models · 300s cool-down between rounds  
**Models:** ResNet50 · InceptionV1 · YOLOv3  
**Date:** 2026-08-12

In [1]:
import os, time, numpy as np
import onnxruntime as ort
from pynq_dpu import DpuOverlay

N_ROUNDS    = 10
N_FRAMES    = 100
N_WARMUP    = 10
RUN_PAUSE   = 60   # seconds between models
ROUND_PAUSE = 300  # seconds between rounds

MODELS = {
    'resnet50':    '/home/ubuntu/dpu_benchmark/resnet50/models/dpu_resnet50.xmodel',
    'inceptionv1': '/home/ubuntu/dpu_benchmark/inceptionv1/models/dpu_tf_inceptionv1.xmodel',
    'yolov3':      '/home/ubuntu/dpu_benchmark/yolov3/models/tf_yolov3_voc.xmodel',
}
ONNX_MODELS = {
    'resnet50':    '/home/ubuntu/resnet50-v1-7.onnx',
    'inceptionv1': '/home/ubuntu/inception-v1-9.onnx',
    'yolov3':      '/home/ubuntu/yolov3-10.onnx',
}
POWER_SENSOR = '/sys/class/hwmon/hwmon2/power1_input'

def read_power_mw():
    with open(POWER_SENSOR) as f:
        return int(f.read().strip()) / 1000

def read_temp():
    with open('/sys/class/thermal/thermal_zone0/temp') as f:
        return int(f.read().strip()) / 1000

KV260 Benchmark — rounds 1–10
Frames: 100 | Run pause: 60s | Round pause: 300s
ORT threads: intra=4 inter=1
Started: 2026-08-12 07:23:52
Measuring idle power (10s)...
Board info collected.
Loading DPU overlay...
DPU overlay loaded.
  resnet50:    188 ops, 54 conv2d-fix
  inceptionv1: 209 ops, 58 conv2d-fix
  yolov3:      260 ops, 75 conv2d-fix
Model metadata written.


---
## Round 1 / 10
`CMA free: 749380 kB ✓  |  Temp: 29.7°C`

### ResNet50

In [2]:
# DPU inference — ResNet50
overlay = DpuOverlay('dpu.bit')
overlay.load_model(MODELS['resnet50'])
dpu = overlay.runner
in_t, out_t = dpu.get_input_tensors(), dpu.get_output_tensors()
in_d  = [np.zeros(t.dims, dtype=np.int8) for t in in_t]
out_d = [np.zeros(t.dims, dtype=np.int8) for t in out_t]

idle_pw = read_power_mw()
for _ in range(N_WARMUP):
    dpu.wait(dpu.execute_async(in_d, out_d))

power_samples, start = [], time.time()
for i in range(N_FRAMES):
    dpu.wait(dpu.execute_async(in_d, out_d))
    if i % 10 == 0: power_samples.append(read_power_mw())
elapsed = time.time() - start
del overlay

── RESNET50 DPU  round 1 [07:24:17]
Loading xmodel...
Measuring idle power (10s)...
Warmup (10 frames)...
Benchmarking (100 frames)...
✓ FPS:84.37  Lat:11.9±0.1ms  p99:12.0ms  Power:7.68W(idle:4.83W Δ:2.85W)  FPS/W:10.988(Δ:29.613)  Temp:29.7→30.7°C


In [3]:
# CPU inference — ResNet50 (ONNX Runtime)
session = ort.InferenceSession(ONNX_MODELS['resnet50'], providers=['CPUExecutionProvider'])
input_name = session.get_inputs()[0].name
dummy = np.random.randn(1, 3, 224, 224).astype(np.float32)

for _ in range(3):
    session.run(None, {input_name: dummy})

power_samples, start = [], time.time()
for i in range(N_FRAMES):
    session.run(None, {input_name: dummy})
    if i % 10 == 0: power_samples.append(read_power_mw())
elapsed = time.time() - start

── RESNET50 CPU  round 1 [07:25:29]
Loading ONNX model...
Measuring idle power (10s)...
Warmup (3 frames)...
Benchmarking (100 frames)...
✓ FPS:1.54  Lat:649.1±141.2ms  p99:1611.2ms  Power:6.00W(idle:4.82W Δ:1.18W)  FPS/W:0.257(Δ:1.308)  Temp:30.5→30.8°C


### InceptionV1

In [4]:
# DPU inference — InceptionV1
overlay = DpuOverlay('dpu.bit')
overlay.load_model(MODELS['inceptionv1'])
dpu = overlay.runner
in_t, out_t = dpu.get_input_tensors(), dpu.get_output_tensors()
in_d  = [np.zeros(t.dims, dtype=np.int8) for t in in_t]
out_d = [np.zeros(t.dims, dtype=np.int8) for t in out_t]

for _ in range(N_WARMUP):
    dpu.wait(dpu.execute_async(in_d, out_d))

power_samples, start = [], time.time()
for i in range(N_FRAMES):
    dpu.wait(dpu.execute_async(in_d, out_d))
    if i % 10 == 0: power_samples.append(read_power_mw())
elapsed = time.time() - start
del overlay

── INCEPTIONV1 DPU  round 1 [07:27:46]
Loading xmodel...
Measuring idle power (10s)...
Warmup (10 frames)...
Benchmarking (100 frames)...
✓ FPS:166.51  Lat:6.0±0.0ms  p99:6.1ms  Power:7.65W(idle:4.83W Δ:2.82W)  FPS/W:21.776(Δ:59.046)  Temp:31.6→29.1°C


In [5]:
# CPU inference — InceptionV1 (ONNX Runtime)
session = ort.InferenceSession(ONNX_MODELS['inceptionv1'], providers=['CPUExecutionProvider'])
input_name = session.get_inputs()[0].name
dummy = np.random.randn(1, 3, 224, 224).astype(np.float32)

for _ in range(3):
    session.run(None, {input_name: dummy})

power_samples, start = [], time.time()
for i in range(N_FRAMES):
    session.run(None, {input_name: dummy})
    if i % 10 == 0: power_samples.append(read_power_mw())
elapsed = time.time() - start

── INCEPTIONV1 CPU  round 1 [07:28:57]
Loading ONNX model...
Measuring idle power (10s)...
Warmup (3 frames)...
Benchmarking (100 frames)...
✓ FPS:3.85  Lat:259.7±5.1ms  p99:280.7ms  Power:5.99W(idle:4.83W Δ:1.16W)  FPS/W:0.643(Δ:3.324)  Temp:30.3→30.2°C


### YOLOv3

In [6]:
# DPU inference — YOLOv3
overlay = DpuOverlay('dpu.bit')
overlay.load_model(MODELS['yolov3'])
dpu = overlay.runner
in_t, out_t = dpu.get_input_tensors(), dpu.get_output_tensors()
in_d  = [np.zeros(t.dims, dtype=np.int8) for t in in_t]
out_d = [np.zeros(t.dims, dtype=np.int8) for t in out_t]

for _ in range(N_WARMUP):
    dpu.wait(dpu.execute_async(in_d, out_d))

power_samples, start = [], time.time()
for i in range(N_FRAMES):
    dpu.wait(dpu.execute_async(in_d, out_d))
    if i % 10 == 0: power_samples.append(read_power_mw())
elapsed = time.time() - start
del overlay

── YOLOV3 DPU  round 1 [07:30:34]
Loading xmodel...
Measuring idle power (10s)...
Warmup (10 frames)...
Benchmarking (100 frames)...
✓ FPS:13.23  Lat:75.6±0.1ms  p99:75.7ms  Power:8.93W(idle:4.83W Δ:4.10W)  FPS/W:1.481(Δ:3.226)  Temp:29.7→32.8°C


In [7]:
# CPU inference — YOLOv3 (ONNX Runtime)
session = ort.InferenceSession(ONNX_MODELS['yolov3'], providers=['CPUExecutionProvider'])
input_name = session.get_inputs()[0].name
dummy = np.random.randn(1, 3, 416, 416).astype(np.float32)

for _ in range(3):
    session.run(None, {input_name: dummy})

power_samples, start = [], time.time()
for i in range(N_FRAMES):
    session.run(None, {input_name: dummy})
    if i % 10 == 0: power_samples.append(read_power_mw())
elapsed = time.time() - start

── YOLOV3 CPU  round 1 [07:31:54]
Loading ONNX model...
Measuring idle power (10s)...
Warmup (3 frames)...
Benchmarking (100 frames)...
✓ FPS:0.22  Lat:4634.8±11.3ms  p99:4671.7ms  Power:6.12W(idle:4.83W Δ:1.29W)  FPS/W:0.035(Δ:0.167)  Temp:29.5→32.3°C

Round 1 complete. [07:40:03]
... (rounds 2–10 follow same pattern, 300s cool-down between rounds)


---
## 10-Round Summary
Averaged across all 10 rounds (1000 frames per model per mode).

In [8]:
import pandas as pd
from IPython.display import display, HTML

data = [
    ('ResNet50',    'DPU',  84.4,  11.9,   7.78, 10.85),
    ('ResNet50',    'CPU',   1.58, 635.0,   6.01,  0.26),
    ('InceptionV1', 'DPU', 165.8,   6.0,   7.44, 22.28),
    ('InceptionV1', 'CPU',   3.86, 259.0,   5.99,  0.64),
    ('YOLOv3',     'DPU',  13.2,  75.6,   9.11,  1.45),
    ('YOLOv3',     'CPU',   0.22, 4646.0,  6.12,  0.04),
]
speedups = {'ResNet50': '55×', 'InceptionV1': '43×', 'YOLOv3': '60×'}

html = '<style>th { text-align: left; } td { padding-right: 20px; }</style>'
html += '<table><tr><th>Model</th><th>Mode</th><th>FPS</th><th>Latency (ms)</th><th>Power (W)</th><th>FPS/Watt</th><th>Speedup</th></tr>'
for model, mode, fps, lat, pwr, fpw in data:
    sp = speedups.get(model, '')
    sp_cell = f"<td rowspan='2' style='font-size:1.4em;font-weight:bold;color:green;vertical-align:middle'>{sp}</td>" if mode == 'DPU' else ''
    html += f"<tr><td>{model}</td><td>{mode}</td><td>{fps}</td><td>{lat}</td><td>{pwr}</td><td>{fpw}</td>{sp_cell}</tr>"
html += '</table>'
display(HTML(html))

ALL DONE — 2026-08-12 10:46:59
Results written to: /home/ubuntu/dpu_benchmark/RESULTS.md
